# DRB2 (A) / DRB4 (B) / ds-RNA (C,D) Domain Contact Analysis — per backend, per interactor couple

**Kernel:** `abcfold-drbs-notebook` (`envs/notebook.yaml`) — install once:
```
conda env create -f envs/notebook.yaml
conda activate abcfold-drbs-notebook
python -m ipykernel install --user --name abcfold-drbs-notebook
```

PLIP contacts from `results/rna_ds_drb2_drb4/all_selected_summary.csv`, produced
by `worflows/postprocessing/Snakefile` (stage 3g `run_plip` + 3h `aggregate`).
This is the **DCL4-dropped** re-run of `rna_ds_dcl4_drb2_drb4` — same DRB2 /
DRB4 / dsRNA sequences, no Dicer. Adapted from
`notebooks/rna_ds_dcl4_drb2_drb4_domain_analysis.ipynb`.

**Scope limitation, important:** `configs/rna_ds_drb2_drb4.yaml`'s `plip.chains`
is `[['A'], ['B', 'C', 'D']]` **with `dnareceptor: true`** — receptor = **DRB2
only**, and the dsRNA (C, D) is folded into the *receptor* group alongside it.
PLIP only ever reports receptor-vs-ligand contacts, so this single PLIP pass
can surface exactly one couple: **DRB2 (A) vs DRB4 (B)**. `reschain` is always
`A`, `reschain_lig` is always `B`; **there are zero DRB2–RNA and zero DRB4–RNA
rows** (verified in the "couples" section below). That is a *configuration*
consequence, not a structural finding — do not read the RNA's absence here as
"the RNA doesn't bind". A dedicated RNA-ligand pass is already scaffolded in
`configs/rna_ds_drb2_drb4.yaml`'s `plip_rna_ligands:` block (`dnareceptor:
false`, no `--chains`) but has **not been run**; when it is, add its couples
here.

**Single pose cluster:** `scripts/pose_cluster_anchor.py` put 599/600 models in
one cluster (silhouette k=2, but the "second" cluster is a single outlier
whose PLIP summary is empty). The DRB2 + DRB4 folded domains converge on one
relative pose across all six backends. So — unlike the DCL4 notebook — there
is **no per-cluster stratification** here; the "per cluster" sections are
replaced by a **per-backend** breakdown and a **residue-level interface map**.

**All six backends ran** for this complex (Chai-1 / Protenix / Boltz all
cleared their token limits at ~900 tokens, vs. hard-fail at the DCL4 complex's
~2600). The energy filter below still removes **RosettaFold3** — its minimised
structures are numerically divergent (final energies up to ~1e19 kJ/mol),
exactly the pathology documented in the DCL4 notebook — leaving **AlphaFold3,
Boltz, Chai-1, OpenFold3, Protenix** (five backends) for the analysis.

---

Domain boundaries (1-based inclusive; residue numbering identical to the
sibling project — sequences copied verbatim, see `configs/rna_ds_drb2_drb4.yaml`).
Originally from UniProt PROSITE annotation. **DRB2's dsRBD2/disordered boundary
is the corrected one** (87-188 / 189-434, not the original PROSITE 87-155 /
156-434) — see `notebooks/drb2_drb4_domain_analysis.ipynb`'s fold-upon-binding
investigation (near-universal cross-backend helix formation + low AIUpred
disorder score, both switching sharply at residue 189). DRB4's boundaries are
unrevised.

**DRB2 (chain A) — receptor**

| DRB2 domain | Residues |
|---|---|
| dsRBD1 | 1-70 |
| linker | 71-86 |
| dsRBD2 | 87-188 (was 87-155 under the original PROSITE call) |
| disordered | 189-434 (was 156-434) |

**DRB4 (chain B) — ligand**

| DRB4 domain | Residues |
|---|---|
| dsRBD1 | 4-73 |
| linker | 74-81 |
| dsRBD2 | 82-150 |
| disordered | 151-291 |
| cryoEM_domain | 292-355 |

**ds-RNA (chains C/D)** — no domains, just nucleotide position (C = 57 nt sense
strand, D = 55 nt antisense strand). No contact rows in this PLIP pass (see
scope limitation above).


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

ROOT = Path("..")
RESULTS_DIR = ROOT / "results" / "rna_ds_drb2_drb4"
RECEPTOR_CHAIN, RECEPTOR_NAME = "A", "DRB2"
LIGAND_CHAIN_NAMES = {"B": "DRB4", "C": "RNA(C)", "D": "RNA(D)"}
FIGURES_DIR = RESULTS_DIR / "figures" / "domain_analysis"

TEMPLATE = "plotly_white"
BACKEND_PALETTE = px.colors.qualitative.Set2
ITYPE_PALETTE = px.colors.qualitative.Set1

def out_path(subdir, filename):
    out_dir = FIGURES_DIR / subdir
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir / filename

def save_fig(fig, filename, subdir=""):
    out = out_path(subdir, filename)
    fig.write_html(out, include_plotlyjs="cdn")
    print(f"Saved: {out}")
    fig.show()

## Load data


In [2]:
csv_path = RESULTS_DIR / "all_selected_summary.csv"
df = pd.read_csv(csv_path)
df = df.rename(columns={"replica": "cluster", "model": "fname"})
df["cluster"] = df["cluster"].astype(int)

sel = pd.read_csv(RESULTS_DIR / "selected_models.csv")
sel["fname"] = sel["staged_cif"].apply(lambda p: Path(p).stem)
sel = sel[["fname", "cluster", "backend", "seed", "sample_index", "ranking_score", "ptm", "iptm"]]

df = df.merge(sel, on=["fname", "cluster"], how="left", validate="many_to_one")
n_missing_meta = df["backend"].isna().sum()
if n_missing_meta:
    print(f"WARNING: {n_missing_meta} contact rows have no matching selected_models.csv entry")

n_models = df.groupby(["cluster", "fname"]).ngroups
print(f"{csv_path}: {len(df)} contact rows, {df['cluster'].nunique()} pose cluster(s), "
      f"{n_models} model(s), before the energy filter below")
print("\ncontact rows per backend:")
print(df.groupby("backend").size().rename("rows").to_frame())

../results/rna_ds_drb2_drb4/all_selected_summary.csv: 37672 contact rows, 1 pose cluster(s), 546 model(s), before the energy filter below

contact rows per backend:
               rows
backend            
alphafold3     2090
boltz          8811
chai1          9562
openfold3      1532
protenix       1718
rosettafold3  13959


## Filter out numerically-unconverged structures

Robust (median / MAD) modified z-score on each minimised structure's final
ChimeraX energy — same approach as `rna_ds_dcl4_drb2_drb4_domain_analysis.ipynb`
and `drb2_drb4_domain_analysis.ipynb`.

Two deviations from the DCL4 notebook, both because this run has a full
six-backend ensemble to protect:

1. **Whole-backend cut:** if a backend has >50 % of its energy-assessed models
   flagged as outliers, the *entire* backend is dropped (all its models,
   assessed or not). This is what removes RosettaFold3.
2. **Missing `_energy.csv` ⇒ keep:** ~10 % of minimisations completed (a
   `*_fixed.pdb` exists) but ChimeraX did not emit a parseable energy
   trajectory. For a *non-dropped* backend those models are kept rather than
   silently discarded — "not assessed", not "bad".


In [3]:
def read_final_energy(energy_csv_path):
    if not energy_csv_path.exists():
        return np.nan
    last_energy = np.nan
    with open(energy_csv_path) as fh:
        next(fh, None)
        for line in fh:
            parts = line.strip().split(",")
            if len(parts) < 2:
                continue
            try:
                last_energy = float(parts[1])
            except ValueError:
                pass
    return last_energy

energy_rows = []
for pdb_path in sorted(RESULTS_DIR.glob("minimized/*/*/*.pdb")):
    if pdb_path.stem.endswith(("_fixed", "_amber", "_nonprot")):
        continue
    cluster = int(pdb_path.parent.parent.name)
    energy_csv = pdb_path.with_name(pdb_path.stem + "_energy.csv")
    energy_rows.append({"fname": pdb_path.stem, "cluster": cluster,
                        "final_energy": read_final_energy(energy_csv)})

energy_all = pd.DataFrame(energy_rows).merge(
    sel[["fname", "cluster", "backend"]], on=["fname", "cluster"], how="left")
energy_df = energy_all.dropna(subset=["final_energy"]).copy()

MOD_Z_THRESHOLD = 3.5
pooled_median = energy_df["final_energy"].median()
pooled_mad = (energy_df["final_energy"] - pooled_median).abs().median()
energy_df["energy_mod_z"] = 0.6745 * (energy_df["final_energy"] - pooled_median) / pooled_mad
energy_df["energy_ok"] = energy_df["energy_mod_z"].abs() <= MOD_Z_THRESHOLD

print(f"Pooled final-energy median={pooled_median:,.0f} kJ/mol, MAD={pooled_mad:,.0f}")
print(f"\nfinal energy by backend (kJ/mol) — assessed models only:")
print(energy_df.groupby("backend")["final_energy"]
      .agg(["count", "min", "median", "max"]).round(1).to_string())

flag_rate = (1 - energy_df.groupby("backend")["energy_ok"].mean()).rename("flagged_frac")
DROP_BACKENDS = sorted(flag_rate[flag_rate > 0.5].index)
print(f"\nflagged fraction by backend (|mod-z| > {MOD_Z_THRESHOLD}):")
print(flag_rate.round(3).to_string())
print(f"\n=> whole-backend drop (>50% flagged): {DROP_BACKENDS or 'none'}")

Pooled final-energy median=3,838 kJ/mol, MAD=4,803

final energy by backend (kJ/mol) — assessed models only:
              count      min        median           max
backend                                                 
alphafold3       99  -4718.8  9.674500e+03  2.791940e+04
boltz            94  -9108.5  1.107600e+03  1.495010e+04
chai1            90  -6972.7  1.853200e+03  3.335460e+04
openfold3        99  -6267.3  3.326000e+03  1.836620e+04
protenix         91 -11008.4  8.352000e+02  1.620430e+04
rosettafold3     66  -6265.0  2.308130e+09  8.537021e+18

flagged fraction by backend (|mod-z| > 3.5):
backend
alphafold3      0.000
boltz           0.000
chai1           0.011
openfold3       0.000
protenix        0.000
rosettafold3    0.833

=> whole-backend drop (>50% flagged): ['rosettafold3']


In [4]:
# keep = (assessed & energy_ok)  OR  (unassessed)  ... minus every dropped backend
assessed_ok = set(zip(energy_df.loc[energy_df["energy_ok"], "cluster"],
                      energy_df.loc[energy_df["energy_ok"], "fname"]))
unassessed = set(zip(energy_all.loc[energy_all["final_energy"].isna(), "cluster"],
                     energy_all.loc[energy_all["final_energy"].isna(), "fname"]))
keep_pairs = assessed_ok | unassessed

drop_fnames = set(sel.loc[sel["backend"].isin(DROP_BACKENDS), "fname"])
df_keys = pd.MultiIndex.from_arrays([df["cluster"], df["fname"]])
n_models_before = df.groupby(["cluster", "fname"]).ngroups

df = df[df_keys.isin(keep_pairs) & ~df["fname"].isin(drop_fnames)].copy()

n_models_after = df.groupby(["cluster", "fname"]).ngroups
print(f"{n_models_after} / {n_models_before} model(s) kept after the energy filter")
print("\nmodels kept per backend:")
print(df.groupby("backend")["fname"].nunique().rename("n_models").to_frame())

455 / 546 model(s) kept after the energy filter

models kept per backend:
            n_models
backend             
alphafold3        99
boltz             99
chai1             99
openfold3         69
protenix          89


## Domain definitions


In [5]:
DRB2_DOMAINS = [
    ("dsRBD1", 1, 70),
    ("linker", 71, 86),
    ("dsRBD2", 87, 188),        # corrected from 87-155 -- see intro cell
    ("disordered", 189, 434),   # corrected from 156-434
]

DRB4_DOMAINS = [
    ("dsRBD1", 4, 73),
    ("linker", 74, 81),
    ("dsRBD2", 82, 150),
    ("disordered", 151, 291),
    ("cryoEM_domain", 292, 355),
]

def make_domain_mapper(domain_ranges):
    """domain_ranges: list of (label, start, end), inclusive both ends.
    Returns a fn mapping a Series of residue numbers -> domain labels;
    residues outside every range become <NA>."""
    intervals = pd.IntervalIndex.from_tuples(
        [(start, end) for _, start, end in domain_ranges], closed="both")
    labels = [label for label, _, _ in domain_ranges]

    def mapper(resnr_series):
        idx = intervals.get_indexer(resnr_series.astype(float))
        return pd.Series([labels[i] if i != -1 else pd.NA for i in idx],
                         index=resnr_series.index, dtype="object")
    return mapper

DRB2_LABELS = [l for l, _, _ in DRB2_DOMAINS]
DRB4_LABELS = [l for l, _, _ in DRB4_DOMAINS]

drb2_mapper = make_domain_mapper(DRB2_DOMAINS)
LIGAND_DOMAINS = {"B": DRB4_DOMAINS}
LIGAND_LABELS  = {"B": DRB4_LABELS}
LIGAND_MAPPERS = {c: make_domain_mapper(d) for c, d in LIGAND_DOMAINS.items()}

# Receptor = DRB2 = chain A. After fix_pdb (pdb4amber + PDBFixer) the whole
# complex is renumbered continuously across chains rather than restarting each
# chain at 1. Verified directly from a 6-model sample spanning every backend
# (identical in all): chain A (DRB2) 1-434, chain B (DRB4) 435-789,
# chain C (RNA sense) 790-846, chain D (RNA antisense) 847-901. So the
# receptor's own resnr is ALREADY 1-based (no offset); only resnr_lig needs
# the per-chain offset subtracted before domain mapping.
df["drb2_domain"] = drb2_mapper(df["resnr"])

CHAIN_OFFSET = {"B": 434, "C": 789, "D": 846}
df["resnr_lig_raw"] = df["resnr_lig"]
for chain, offset in CHAIN_OFFSET.items():
    mask = df["reschain_lig"] == chain
    df.loc[mask, "resnr_lig"] = df.loc[mask, "resnr_lig"] - offset

for chain, expected_len in [("B", 355), ("C", 57), ("D", 55)]:
    sub = df[df["reschain_lig"] == chain]
    if sub.empty:
        print(f"chain {chain}: no rows in this PLIP pass")
        continue
    bad = sub[~sub["resnr_lig"].between(1, expected_len)]
    if len(bad):
        print(f"WARNING: {len(bad)} chain-{chain} rows outside [1,{expected_len}] "
              f"after offset -- re-verify CHAIN_OFFSET['{chain}'].")
    else:
        print(f"chain {chain}: offset {CHAIN_OFFSET[chain]} verified OK "
              f"(all {len(sub)} rows land in [1,{expected_len}])")

df["ligand_domain"] = pd.NA
for chain, mapper in LIGAND_MAPPERS.items():
    mask = df["reschain_lig"] == chain
    df.loc[mask, "ligand_domain"] = mapper(df.loc[mask, "resnr_lig"])

n_unmapped_r = df["drb2_domain"].isna().sum()
print(f"\nUnmapped DRB2 residues (outside any domain): {n_unmapped_r} "
      f"({100*n_unmapped_r/len(df):.1f}%)")
df[["resnr", "drb2_domain", "reschain_lig", "resnr_lig_raw", "resnr_lig", "ligand_domain"]].head(5)

chain B: offset 434 verified OK (all 23607 rows land in [1,355])
chain C: no rows in this PLIP pass
chain D: no rows in this PLIP pass



Unmapped DRB2 residues (outside any domain): 0 (0.0%)


,resnr,drb2_domain,reschain_lig,resnr_lig_raw,resnr_lig,ligand_domain
0,102,dsRBD2,B,694,260,disordered
1,109,dsRBD2,B,624,190,disordered
2,109,dsRBD2,B,622,188,disordered
3,133,dsRBD2,B,761,327,cryoEM_domain
4,155,dsRBD2,B,781,347,cryoEM_domain


## Interactor couples in this dataset

Every heatmap in this notebook is one **couple**: DRB2 (fixed receptor) vs. one
partner. Three couples are conceivable (DRB2-DRB4, DRB2-RNA(C), DRB2-RNA(D)),
but per the scope limitation in the intro only **DRB2-DRB4** can appear — the
check below makes that explicit so an empty couple later is expected, not a
surprise.


In [6]:
COUPLES = ["B", "C", "D"]   # DRB4, RNA(C), RNA(D)

print("Contact rows per couple (DRB2 vs. partner):")
for c in COUPLES:
    n = (df["reschain_lig"] == c).sum()
    n_models_c = df.loc[df["reschain_lig"] == c, ["cluster", "fname"]].drop_duplicates().shape[0]
    status = "POPULATED" if n else "EMPTY -- no contacts in this PLIP pass (see intro)"
    print(f"  {RECEPTOR_NAME} x {LIGAND_CHAIN_NAMES[c]:8s}: {n:6d} contact rows "
          f"across {n_models_c:3d} model(s) -- {status}")

Contact rows per couple (DRB2 vs. partner):
  DRB2 x DRB4    :  23607 contact rows across 455 model(s) -- POPULATED
  DRB2 x RNA(C)  :      0 contact rows across   0 model(s) -- EMPTY -- no contacts in this PLIP pass (see intro)
  DRB2 x RNA(D)  :      0 contact rows across   0 model(s) -- EMPTY -- no contacts in this PLIP pass (see intro)


In [7]:
def domain_pair_heatmap(data, ligand_chain, title, filename, n_models_norm, subdir="domain_contacts"):
    ligand_name = LIGAND_CHAIN_NAMES[ligand_chain]
    labels = LIGAND_LABELS[ligand_chain]
    sub = data[data["reschain_lig"] == ligand_chain].dropna(subset=["drb2_domain", "ligand_domain"])
    if sub.empty:
        print(f"No {RECEPTOR_NAME}-{ligand_name} contacts in this subset -- skipping '{title}'.")
        return None

    ct = (sub.groupby(["drb2_domain", "ligand_domain"], observed=True).size()
          .unstack(fill_value=0).reindex(index=DRB2_LABELS, columns=labels, fill_value=0))
    rate = ct / n_models_norm if n_models_norm else ct

    fig = go.Figure(go.Heatmap(
        z=rate.values, x=labels, y=DRB2_LABELS, colorscale="YlOrRd",
        text=[[f"{v:.2f}" if v > 0 else "" for v in row] for row in rate.values],
        texttemplate="%{text}", textfont=dict(size=9),
        hovertemplate=f"{RECEPTOR_NAME} domain: %{{y}}<br>{ligand_name} domain: %{{x}}"
                      "<br>%{z:.3f} contacts/model<extra></extra>",
        colorbar=dict(title="Mean<br>contacts/<br>model"),
    ))
    fig.update_layout(
        title=title, xaxis_title=f"{ligand_name} domain", yaxis_title=f"{RECEPTOR_NAME} domain",
        yaxis=dict(autorange="reversed"), template=TEMPLATE,
        width=max(500, len(labels) * 110), height=max(420, len(DRB2_LABELS) * 70),
    )
    save_fig(fig, filename, subdir)
    return ct

def rna_contact_heatmap(data, strand, n_models_norm, title=None, filename=None, subdir="domain_contacts"):
    sub = data[data["reschain_lig"] == strand].dropna(subset=["drb2_domain"])
    if sub.empty:
        print(f"No {RECEPTOR_NAME}-RNA({strand}) contacts in this subset -- skipping.")
        return None
    nt_positions = sorted(sub["resnr_lig"].dropna().unique())
    ct = (sub.groupby(["resnr_lig", "drb2_domain"], observed=True).size()
          .unstack(fill_value=0).reindex(index=nt_positions, columns=DRB2_LABELS, fill_value=0))
    rate = ct / n_models_norm if n_models_norm else ct
    fig = go.Figure(go.Heatmap(
        z=rate.values, x=DRB2_LABELS, y=[str(p) for p in nt_positions], colorscale="YlOrRd",
        hovertemplate=(f"{RECEPTOR_NAME} domain: %{{x}}<br>RNA nt (strand {strand}): %{{y}}"
                       "<br>%{z:.3f} contacts/model<extra></extra>"),
        colorbar=dict(title="Contacts<br>/ model"),
    ))
    fig.update_layout(
        title=title or f"{RECEPTOR_NAME} x RNA strand {strand} contact rate",
        xaxis_title=f"{RECEPTOR_NAME} domain", yaxis_title=f"RNA nt position (strand {strand})",
        yaxis=dict(autorange="reversed", tickfont=dict(size=8)),
        template=TEMPLATE, width=900, height=max(400, len(nt_positions) * 14),
    )
    save_fig(fig, filename or f"drb2_rna_{strand}_heatmap.html", subdir)
    return ct

def couple_heatmap(data, ligand_chain, n_models_norm, title, filename, subdir="domain_contacts"):
    """domain x domain for protein partners (B), nt-position x domain for RNA (C, D)."""
    if ligand_chain == "B":
        return domain_pair_heatmap(data, ligand_chain, title, filename, n_models_norm, subdir)
    return rna_contact_heatmap(data, ligand_chain, n_models_norm, title, filename, subdir)

## One heatmap per couple (all backends pooled)


In [8]:
n_models_total = df.groupby(["cluster", "fname"]).ngroups
ct_all = {}
for c in COUPLES:
    ct_all[c] = couple_heatmap(
        df, c, n_models_total,
        f"{RECEPTOR_NAME} x {LIGAND_CHAIN_NAMES[c]} -- all backends pooled (n={n_models_total} models)",
        f"drb2_{LIGAND_CHAIN_NAMES[c].lower()}_heatmap_pooled.html",
    )

Saved: ../results/rna_ds_drb2_drb4/figures/domain_analysis/domain_contacts/drb2_drb4_heatmap_pooled.html


No DRB2-RNA(C) contacts in this subset -- skipping.
No DRB2-RNA(D) contacts in this subset -- skipping.


## Interaction-type breakdown (all backends pooled)


In [9]:
print("Interaction type counts:")
display(df["interaction_type"].value_counts().rename("count").to_frame())

print("\nInteraction type by receptor-ligand chain pair:")
display(
    df.groupby(["reschain", "reschain_lig", "interaction_type"], observed=True)
    .size().rename("count").reset_index().sort_values("count", ascending=False).head(20)
)

# contacts/model per interaction type, by backend -- does the ensemble agree?
per_backend_itype = (
    df.groupby(["backend", "interaction_type"], observed=True).size()
      .div(df.groupby("backend")["fname"].nunique(), level="backend")
      .rename("contacts_per_model").reset_index()
)
fig = px.bar(per_backend_itype, x="backend", y="contacts_per_model", color="interaction_type",
             color_discrete_sequence=ITYPE_PALETTE, template=TEMPLATE,
             title=f"{RECEPTOR_NAME} x DRB4 contacts per model, by interaction type and backend")
fig.update_layout(width=760, height=460, xaxis_title="", yaxis_title="contacts / model")
save_fig(fig, "drb2_drb4_interaction_types_by_backend.html")

Interaction type counts:


,count
interaction_type,
hydrogen_bonds,12603
hydrophobic_interactions,8518
salt_bridges,2328
pi-cation_interactions,131
pi-stacking,27



Interaction type by receptor-ligand chain pair:


,reschain,reschain_lig,interaction_type,count
0,A,B,hydrogen_bonds,12603
1,A,B,hydrophobic_interactions,8518
4,A,B,salt_bridges,2328
2,A,B,pi-cation_interactions,131
3,A,B,pi-stacking,27


Saved: ../results/rna_ds_drb2_drb4/figures/domain_analysis/drb2_drb4_interaction_types_by_backend.html


## Per-backend breakdown

There is only one pose cluster, so instead of the DCL4 notebook's per-cluster
sections this is a **per-backend** comparison: one panel per couple, one
heatmap per surviving backend, shared colour scale within the panel. This is
the cross-architecture agreement check — do independent models place the same
domain-domain contacts?


In [10]:
backends = sorted(df["backend"].dropna().unique())
backend_n_models = df.groupby("backend")["fname"].nunique()
print(f"{len(backends)} backend(s) after the energy filter: " +
      ", ".join(f"{b} (n={backend_n_models[b]})" for b in backends))

5 backend(s) after the energy filter: alphafold3 (n=99), boltz (n=99), chai1 (n=99), openfold3 (n=69), protenix (n=89)


In [11]:
def couple_heatmap_matrix(data, ligand_chain, n_models_norm):
    """Same content as couple_heatmap() but returns (x_labels, y_labels, rate matrix)
    without plotting -- for multi-panel comparisons."""
    if ligand_chain == "B":
        labels = LIGAND_LABELS[ligand_chain]
        sub = data[data["reschain_lig"] == ligand_chain].dropna(subset=["drb2_domain", "ligand_domain"])
        if sub.empty:
            return None
        ct = (sub.groupby(["drb2_domain", "ligand_domain"], observed=True).size()
              .unstack(fill_value=0).reindex(index=DRB2_LABELS, columns=labels, fill_value=0))
        y_labels = DRB2_LABELS
    else:
        sub = data[data["reschain_lig"] == ligand_chain].dropna(subset=["drb2_domain"])
        if sub.empty:
            return None
        nt_positions = sorted(sub["resnr_lig"].dropna().unique())
        ct = (sub.groupby(["resnr_lig", "drb2_domain"], observed=True).size()
              .unstack(fill_value=0).reindex(index=nt_positions, columns=DRB2_LABELS, fill_value=0))
        labels, y_labels = DRB2_LABELS, [str(p) for p in nt_positions]
    return labels, y_labels, (ct / n_models_norm if n_models_norm else ct)

def backend_panel_for_couple(ligand_chain):
    ligand_name = LIGAND_CHAIN_NAMES[ligand_chain]
    per_backend = {}
    for b in backends:
        res = couple_heatmap_matrix(df[df["backend"] == b], ligand_chain, backend_n_models[b])
        if res is not None:
            per_backend[b] = res
    if not per_backend:
        print(f"No {RECEPTOR_NAME}-{ligand_name} contacts for any backend -- skipping panel.")
        return

    present = list(per_backend.keys())
    zmax = max(rate.values.max() for _, _, rate in per_backend.values())
    ncols = min(3, len(present))
    nrows = -(-len(present) // ncols)

    fig = make_subplots(rows=nrows, cols=ncols,
                        subplot_titles=[f"{b} (n={backend_n_models[b]})" for b in present],
                        shared_yaxes=True)
    for i, b in enumerate(present):
        labels, y_labels, rate = per_backend[b]
        row, col = i // ncols + 1, i % ncols + 1
        fig.add_trace(go.Heatmap(
            z=rate.values, x=labels, y=y_labels, colorscale="YlOrRd",
            zmin=0, zmax=zmax, showscale=(b == present[-1]),
            text=[[f"{v:.2f}" if v > 0 else "" for v in r] for r in rate.values],
            texttemplate="%{text}", textfont=dict(size=8),
            hovertemplate=f"{b}<br>{RECEPTOR_NAME}: %{{y}}<br>{ligand_name}: %{{x}}"
                          "<br>%{z:.3f} contacts/model<extra></extra>",
            colorbar=dict(title="Mean<br>contacts/<br>model"),
        ), row=row, col=col)
    fig.update_yaxes(autorange="reversed")
    fig.update_layout(title=f"{RECEPTOR_NAME} x {ligand_name} domain contact pairs by backend",
                      template=TEMPLATE, width=max(760, 380 * ncols),
                      height=max(430, 150 * nrows))
    save_fig(fig, f"drb2_{ligand_name.lower()}_heatmap_by_backend.html", "per_backend")

for c in COUPLES:
    backend_panel_for_couple(c)

Saved: ../results/rna_ds_drb2_drb4/figures/domain_analysis/per_backend/drb2_drb4_heatmap_by_backend.html


No DRB2-RNA(C) contacts for any backend -- skipping panel.
No DRB2-RNA(D) contacts for any backend -- skipping panel.


## Residue-level interface map

Domain x domain is coarse when the whole ensemble converges on **one**
interface. These panels drop to single-residue resolution: contact rate
(contacts per model) for the busiest DRB2 and DRB4 residues, split by
interaction type, plus the 2-D DRB2-residue x DRB4-residue contact map — the
actual predicted interface footprint, pooled over all five backends.


In [12]:
TOP_N = 25
bc = df[df["reschain_lig"] == "B"].copy()
n_norm = df.groupby(["cluster", "fname"]).ngroups

def residue_rate_bar(data, resnr_col, restype_col, chain_label, filename):
    per_res_total = (data.groupby([resnr_col, restype_col], observed=True).size()
                     .div(n_norm).rename("rate").reset_index()
                     .sort_values("rate", ascending=False).head(TOP_N))
    order = per_res_total[resnr_col].tolist()
    per_res_itype = (data.groupby([resnr_col, restype_col, "interaction_type"], observed=True).size()
                     .div(n_norm).rename("rate").reset_index())
    per_res_itype = per_res_itype[per_res_itype[resnr_col].isin(order)].copy()
    per_res_itype["label"] = (per_res_itype[restype_col].astype(str)
                              + per_res_itype[resnr_col].astype(int).astype(str))
    label_order = [f"{per_res_total.loc[per_res_total[resnr_col] == r, restype_col].iloc[0]}{int(r)}"
                   for r in order]
    fig = px.bar(per_res_itype, x="label", y="rate", color="interaction_type",
                 category_orders={"label": label_order},
                 color_discrete_sequence=ITYPE_PALETTE, template=TEMPLATE,
                 title=f"{chain_label} interface residues -- top {TOP_N} by contact rate "
                       f"(pooled, n={n_norm} models)")
    fig.update_layout(width=950, height=460, xaxis_title=f"{chain_label} residue",
                      yaxis_title="contacts / model", xaxis_tickangle=-45)
    save_fig(fig, filename, "residue_interface")
    return per_res_total

top_drb2 = residue_rate_bar(bc, "resnr", "restype", "DRB2", "drb2_top_residues.html")
top_drb4 = residue_rate_bar(bc, "resnr_lig", "restype_lig", "DRB4", "drb4_top_residues.html")
display(top_drb2.rename(columns={"resnr": "drb2_resnr", "restype": "drb2_restype"}).reset_index(drop=True))
display(top_drb4.rename(columns={"resnr_lig": "drb4_resnr", "restype_lig": "drb4_restype"}).reset_index(drop=True))

Saved: ../results/rna_ds_drb2_drb4/figures/domain_analysis/residue_interface/drb2_top_residues.html


Saved: ../results/rna_ds_drb2_drb4/figures/domain_analysis/residue_interface/drb4_top_residues.html


,drb2_resnr,drb2_restype,rate
0,357,ARG,0.714286
1,355,ARG,0.707692
2,342,ARG,0.567033
3,348,PHE,0.503297
4,428,ARG,0.496703
5,110,ARG,0.461538
6,411,ARG,0.448352
7,334,ARG,0.437363
8,105,ARG,0.437363
9,434,ILE,0.430769


,drb4_resnr,drb4_restype,rate
0,323,VAL,0.751648
1,342,PHE,0.745055
2,347,PHE,0.727473
3,346,LYS,0.690110
4,326,ARG,0.681319
5,317,ILE,0.549451
6,283,TRP,0.514286
7,275,ARG,0.494505
8,314,HIS,0.457143
9,319,THR,0.454945


In [13]:
# 2-D interface contact map: DRB2 residue x DRB4 residue, contact rate, pooled.
top_drb2_res = top_drb2["resnr"].tolist()
top_drb4_res = top_drb4["resnr_lig"].tolist()
pair = bc[bc["resnr"].isin(top_drb2_res) & bc["resnr_lig"].isin(top_drb4_res)]
mat = (pair.groupby(["resnr", "resnr_lig"], observed=True).size()
       .div(n_norm).unstack(fill_value=0)
       .reindex(index=sorted(top_drb2_res), columns=sorted(top_drb4_res), fill_value=0))
drb2_tick = {int(r): f"{bc.loc[bc['resnr']==r,'restype'].iloc[0]}{int(r)}" for r in mat.index}
drb4_tick = {int(c): f"{bc.loc[bc['resnr_lig']==c,'restype_lig'].iloc[0]}{int(c)}" for c in mat.columns}
fig = go.Figure(go.Heatmap(
    z=mat.values,
    x=[drb4_tick[c] for c in mat.columns], y=[drb2_tick[r] for r in mat.index],
    colorscale="YlOrRd",
    hovertemplate="DRB2 %{y}<br>DRB4 %{x}<br>%{z:.3f} contacts/model<extra></extra>",
    colorbar=dict(title="contacts<br>/ model")))
fig.update_layout(title=f"DRB2 x DRB4 residue-residue contact map -- top {TOP_N} each, pooled (n={n_norm})",
                  xaxis_title="DRB4 residue", yaxis_title="DRB2 residue",
                  yaxis=dict(autorange="reversed"), template=TEMPLATE, width=900, height=760)
save_fig(fig, "drb2_drb4_residue_contact_map.html", "residue_interface")

Saved: ../results/rna_ds_drb2_drb4/figures/domain_analysis/residue_interface/drb2_drb4_residue_contact_map.html

## Export contact tables


In [17]:
# pooled + per-backend domain-pair contact-rate matrices, and residue tables
export_dir = out_path("tables", "")
for b in ["_pooled"] + backends:
    data = df if b == "_pooled" else df[df["backend"] == b]
    n_norm_b = data.groupby(["cluster", "fname"]).ngroups
    res = couple_heatmap_matrix(data, "B", n_norm_b)
    if res is None:
        continue
    _, _, rate = res
    tag = "pooled" if b == "_pooled" else b
    p = export_dir / f"drb2_drb4_domain_pair_rate_{tag}.csv"
    rate.to_csv(p)
    print(f"Saved: {p}")

top_drb2.assign(chain="DRB2").rename(columns={"resnr": "resnr", "restype": "restype"}) \
    .to_csv(export_dir / "drb2_top_interface_residues.csv", index=False)
top_drb4.assign(chain="DRB4").rename(columns={"resnr_lig": "resnr", "restype_lig": "restype"}) \
    .to_csv(export_dir / "drb4_top_interface_residues.csv", index=False)
print(f"Saved: {export_dir/'drb2_top_interface_residues.csv'}")
print(f"Saved: {export_dir/'drb4_top_interface_residues.csv'}")

Saved: ../results/rna_ds_drb2_drb4/figures/domain_analysis/tables/drb2_drb4_domain_pair_rate_pooled.csv
Saved: ../results/rna_ds_drb2_drb4/figures/domain_analysis/tables/drb2_drb4_domain_pair_rate_alphafold3.csv
Saved: ../results/rna_ds_drb2_drb4/figures/domain_analysis/tables/drb2_drb4_domain_pair_rate_boltz.csv
Saved: ../results/rna_ds_drb2_drb4/figures/domain_analysis/tables/drb2_drb4_domain_pair_rate_chai1.csv
Saved: ../results/rna_ds_drb2_drb4/figures/domain_analysis/tables/drb2_drb4_domain_pair_rate_openfold3.csv
Saved: ../results/rna_ds_drb2_drb4/figures/domain_analysis/tables/drb2_drb4_domain_pair_rate_protenix.csv
Saved: ../results/rna_ds_drb2_drb4/figures/domain_analysis/tables/drb2_top_interface_residues.csv
Saved: ../results/rna_ds_drb2_drb4/figures/domain_analysis/tables/drb4_top_interface_residues.csv
